# ColabUMA TS — transition state search with UMA

Searches for the **transition state (TS)** of a reaction with Meta's machine learning potential **UMA**. No coding needed.

1. **DMF** (Direct MaxFlux method, [PyDMF](https://github.com/shin1koda/dmf)) optimizes the reaction path between the reactant and the product; the highest-energy point on the path becomes the initial TS guess
2. **Sella** optimizes it to a first-order saddle point (TS). The initial Hessian comes from numerical differentiation of UMA
3. A vibrational analysis checks that there is exactly one imaginary frequency; the TS is then displaced slightly both ways along the imaginary mode and optimized, to check that it connects the reactant and the product (simple IRC check)
4. **DFT single points** ([GPU4PySCF](https://github.com/pyscf/gpu4pyscf)) of the reactant, TS and product at the UMA structures give the activation energies to compare

## How to use

1. In **Runtime → Change runtime type**, choose **T4 GPU**
2. First time only: open 🔑 (Secrets) in the left sidebar, add your Hugging Face access token with the name `HF_TOKEN`, and turn on notebook access
3. In **1. Input form** below, set the output name, charge, spin multiplicity, how to build the initial TS guess, and the DFT settings
4. Choose **Runtime → Run all**
5. In **2. File selection and checks**, choose the reactant file and then the product file (xyz or gjf). With "TS guess (Sella only)", choose one file with the initial TS guess. You can leave it running after that
6. Check the path, the TS, the imaginary frequency and the energies in the cells below, and download the ZIP of results in the last cell

**The reactant and the product must have the same atoms in the same order** (DMF matches atoms by number). The scan results (`preopt.xyz` and `postopt.xyz`) work as they are.

> **Before you start**: the model weights, [facebook/UMA](https://huggingface.co/facebook/UMA), require an access request. Log in to Hugging Face, accept the FAIR Chemistry License v1 on that page, and once approved, create a token with Read access under [Access Tokens](https://huggingface.co/settings/tokens).

## Output (contents of the ZIP)

| File | Contents |
|---|---|
| `<output name>_ts.xyz` | TS optimized with Sella |
| `<output name>_ts_opt.log` | Path of the TS optimization (in the format of Gaussian's opt=ts) |
| `<output name>_ts_freq.log` | Frequencies and normal modes of the TS (in the format of Gaussian's freq; GaussView animates the imaginary mode) |
| `<output name>_ts_freq.txt` | List of frequencies |
| `<output name>_ts.gjf` | Gaussian input for checking the TS with DFT (the route line comes from the form) |
| `<output name>_dmf_path.xyz` / `.log` / `.csv` | Reaction path optimized with DMF (31 evenly spaced points plus the maximum) and energies |
| `<output name>_dmf_init.xyz` | Initial path built with FB-ENM |
| `<output name>_ts_guess.xyz` | Highest point on the DMF path (Sella's starting structure) |
| `<output name>_reactant.xyz` / `product.xyz` | Reactant and product (the optimized ones if the endpoints were optimized, with `_opt.log`) |
| `<output name>_irc_minus.xyz` / `irc_plus.xyz` | Structures optimized after a displacement along the imaginary mode |
| `<output name>_dft.csv` | DFT single point energies |
| `<output name>_log.txt` | Input settings, software versions, results of each stage |

## Notes

- The TS found is a TS on the UMA potential energy surface. The DFT single points use the UMA structures, so the DFT TS lies slightly elsewhere. Finish with a DFT TS optimization and IRC, for example with the gjf in the ZIP.
- The "simple IRC check" is not a real IRC. It displaces the atom that moves most along the imaginary mode by 0.15 Å, optimizes, and checks whether the bonding matches the reactant and the product.
- The charge and spin multiplicity are fixed throughout. Open-shell DFT uses UKS.
- DFT runs on GPU4PySCF (density fitting). The T4 is slow at double precision, so large molecules and def2-TZVPD take time. Without a GPU, DFT runs on the CPU with PySCF.
- The first run takes a few minutes to install, including building IPOPT for DMF.
- UMA is a machine learning potential, not a replacement for DFT.

## License

- This notebook and the shared code (`colabuma`): [MIT License](https://github.com/RChibaChiba/ColabChem/blob/main/LICENSE)
- UMA model weights: [FAIR Chemistry License v1](https://huggingface.co/facebook/UMA)
- Libraries installed at runtime are used under their own licenses. See [License in the README](https://github.com/RChibaChiba/ColabChem/blob/main/README.md#license)

## Citation

If you use this notebook in a publication, please cite:

- B. M. Wood *et al.*, "UMA: A Family of Universal Models for Atoms", arXiv:2506.23971 (2025).
- D. S. Levine *et al.*, "The Open Molecules 2025 (OMol25) Dataset, Evaluations, and Models", arXiv:2505.08762 (2025).
- fairchem: https://github.com/facebookresearch/fairchem

If you use the transition state search (DMF, Sella), please also cite:

- S.-i. Koda and S. Saito, *J. Open Source Softw.* **11**, 10379 (2026). (PyDMF)
- S.-i. Koda and S. Saito, *J. Chem. Theory Comput.* **20**, 2798–2811 (2024). (Direct MaxFlux)
- S.-i. Koda and S. Saito, *J. Chem. Theory Comput.* **20**, 7176–7187 (2024); **21**, 3513–3522 (2025). (FB-ENM, CFB-ENM)
- E. D. Hermes, K. Sargsyan, H. N. Najm, J. Zádor, *J. Chem. Theory Comput.* **18**, 6974–6988 (2022). (Sella)
- If you use the DFT single points: PySCF (Q. Sun *et al.*, *J. Chem. Phys.* **153**, 024109 (2020)) and GPU4PySCF (X. Wu *et al.*, arXiv:2404.09452 (2024))

In [ ]:
#@title 1. Input form { display-mode: "form" }
#@markdown When the form is filled in, choose **Runtime → Run all**. Pick the files in the next cell; after that you can leave it running.

#@markdown ### Output
output_name = "result"  #@param {type:"string"}

#@markdown ### Structure
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}
#@markdown Build the initial TS guess from a reactant and product with DMF, or optimize your own TS guess with Sella only.
ts_input = "reactant and product (DMF → Sella)"  #@param ["reactant and product (DMF → Sella)", "TS guess (Sella only)"]
#@markdown Optimize the reactant and product without constraints before DMF.
opt_endpoints = True  #@param {type:"boolean"}

#@markdown ### DMF (reaction path and initial TS guess)
#@markdown Number of points along the path where the energy is computed. More points give a better TS guess but take longer.
dmf_nmove = 5  #@param {type:"integer"}
dmf_convergence = "middle"  #@param ["loose", "middle", "tight"]
dmf_max_iter = 300  #@param {type:"integer"}

#@markdown ### Sella (TS optimization)
#@markdown Build the initial Hessian by numerical differentiation of UMA (6N force calls). Off starts from an approximate Hessian.
ts_exact_hessian = True  #@param {type:"boolean"}
#@markdown Displace the TS slightly along the imaginary mode, optimize both sides, and check where they lead (simple IRC check).
check_connection = True  #@param {type:"boolean"}

#@markdown ### DFT single point (gpu4pyscf)
#@markdown Computes DFT energies at the UMA structures and compares activation energies. ωB97M-V/def2-TZVPD is the level of theory of UMA's training data (OMol25). def2-TZVPD can take tens of minutes per point on a T4.
run_dft = True  #@param {type:"boolean"}
dft_functional = "B3LYP-D3(BJ)"  #@param ["ωB97M-V (same as OMol25)", "B3LYP-D3(BJ)", "PBE0-D3(BJ)", "M06-2X", "r2SCAN-D4"]
dft_basis = "def2-SVP"  #@param ["def2-SVP", "def2-TZVP", "def2-TZVPD"]
#@markdown Time limit for DFT (minutes, all structures together). Past it, DFT is skipped and the UMA results are saved. 0 means no limit (DFT still stops before the Colab runtime is likely to end).
dft_time_limit = 60  #@param {type:"integer"}
#@markdown Route line of the Gaussian TS gjf put in the ZIP.
gaussian_route = "# opt=(ts,calcfc,noeigentest) freq b3lyp/def2svp empiricaldispersion=gd3bj"  #@param {type:"string"}

#@markdown ### Settings
model_name = "uma-s-1p1"  #@param ["uma-s-1p1", "uma-s-1p2", "uma-m-1p1"]
task_name = "omol"  #@param ["omol"]
optimizer_name = "LBFGS"  #@param ["LBFGS", "BFGS", "FIRE"]
#@markdown Convergence threshold fmax (eV/Å). normal is close to Gaussian's default, tight to Gaussian's opt=tight.
convergence = "normal (0.02)"  #@param ["loose (0.05)", "normal (0.02)", "tight (0.001)", "custom"]
#@markdown fmax (eV/Å) used when custom is chosen.
fmax_custom = 0.02  #@param {type:"number"}
#@markdown The convergence threshold and maximum steps apply to the endpoint optimization, the TS optimization and the simple IRC check.
max_steps = 500  #@param {type:"integer"}

In [ ]:
#@title 2. File selection and checks { display-mode: "form" }
#@markdown File selection and the token check happen here. You can leave the install and calculations after this running.

import os
import subprocess
import sys

COLABUMA_REF = "main"  # GitHub branch or tag of the shared code (colabuma package)


def _use_colabuma(ref):
    # Use colabuma next to the notebook if it is there, otherwise fetch it from GitHub
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabChem" if os.path.isdir("/content") else os.path.expanduser("~/ColabChem")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabChem.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"Could not fetch the ColabChem shared code from GitHub (ref {ref}).\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import dft, env, gaussian, opt, output, structure, ts, ui
from colabuma.ui import ColabUMAError, ok

ui.install_error_handler()

fmax = opt.fmax_from_form(convergence, fmax_custom)
structure.check_form(fmax, max_steps, multiplicity)
job = opt.Job(structure.out_prefix(output_name), charge, multiplicity, model_name, task_name,
              optimizer_name, fmax, max_steps, convergence)
use_dmf = ts_input.startswith(("reactant", "反応物"))
if dmf_nmove < 1 or dmf_max_iter < 1:
    raise ColabUMAError("DMF's nmove and maximum iterations must be 1 or more.", "Typical values are nmove = 5 and 300 iterations.",
                        ja="DMF の nmove と最大反復回数は1以上にしてください。", ja_hint="通常は nmove = 5、最大反復回数 = 300 程度です。")
if run_dft:
    dft.parse_method(dft_functional, dft_basis)
    if dft_time_limit < 0:
        raise ColabUMAError("The DFT time limit must be 0 or more.", "0 means no limit.",
                            ja="DFT の時間の上限は0以上にしてください。", ja_hint="0 なら上限なしです。")
env.check_gpu()
env.ensure_token()
if use_dmf:
    reactant_file = structure.upload("reactant", "the reactant xyz or gjf file")
    product_file = structure.upload("product", "the product xyz or gjf file")
    _n = [structure.quick_atom_count(*reactant_file), structure.quick_atom_count(*product_file)]
    if all(_n) and _n[0] != _n[1]:
        raise ColabUMAError(f"The reactant ({_n[0]} atoms) and product ({_n[1]} atoms) have different numbers of atoms.",
                            "Choose two structures of the same molecule with the atoms in the same order.",
                            ja=f"反応物（{_n[0]} 原子）と生成物（{_n[1]} 原子）の原子数が違います。",
                            ja_hint="同じ分子の、原子の順番がそろった2つの構造を選んでください。")
else:
    guess_file = structure.upload("guess", "the xyz or gjf file of the initial TS guess")
ok("Input checked. You can leave it running from here.")

In [ ]:
#@title 3. Setup (install and login) { display-mode: "form" }
DEVICE = env.setup(["ts", "dft"] if run_dft else ["ts"])

In [ ]:
#@title 4. Load the model { display-mode: "form" }
calc = env.load_calculator(model_name, task_name)

In [ ]:
#@title 5. Prepare the structures { display-mode: "form" }
if use_dmf:
    reactant0, _desc_r = structure.read_structure(*reactant_file, charge, multiplicity)
    product0, _desc_p = structure.read_structure(*product_file, charge, multiplicity)
    product0 = ts.check_endpoints(reactant0, product0)
    source_desc = f"reactant {_desc_r} / product {_desc_p}"
    formed, broken = ts.bond_changes(reactant0, product0)
    print("Reactant (numbered; red: atoms whose bonds change):")
    structure.show_numbered(reactant0, {i for b in formed + broken for i in b})
    print(f"Bonds formed: {ts.bond_text(formed, reactant0)}  bonds broken: {ts.bond_text(broken, reactant0)}")
    ok(f"Reactant and product ready: {reactant0.get_chemical_formula()} ({len(reactant0)} atoms).")
else:
    guess0, source_desc = structure.read_structure(*guess_file, charge, multiplicity)
    structure.show_numbered(guess0)
    ok(f"Initial TS guess ready: {guess0.get_chemical_formula()} ({len(guess0)} atoms).")
if run_dft:
    dft.check_memory([("", reactant0 if use_dmf else guess0)], charge, multiplicity,
                     dft.parse_method(dft_functional, dft_basis)[1])

In [ ]:
#@title 6. Reaction path (DMF) { display-mode: "form" }
#@markdown The DMF Converge bar shows roughly how far IPOPT's dual infeasibility has come, in orders of magnitude, toward the threshold.
endpoints = dmf_res = None
reactant = product = None
if use_dmf:
    if opt_endpoints:
        print(f"Optimizing the reactant and product with {optimizer_name} (fmax < {fmax} eV/Å)")
        endpoints = {"reactant": opt.optimize(reactant0, calc, job, "Reactant"),
                     "product": opt.optimize(product0, calc, job, "Product")}
        for _r in endpoints.values():
            opt.warn_unconverged(_r)
        reactant = endpoints["reactant"]["atoms"]
        product = ts.check_endpoints(reactant, endpoints["product"]["atoms"])
        endpoints["product"]["atoms"] = product
    else:
        reactant, product = reactant0, product0
    dmf_res = ts.run_dmf(reactant, product, calc, job, nmove=dmf_nmove, tol=dmf_convergence, max_iter=dmf_max_iter)
    if endpoints is None:
        endpoints = {"reactant": {"atoms": reactant, "energy": dmf_res["energies"][0]},
                     "product": {"atoms": product, "energy": dmf_res["energies"][-1]}}
    ts_guess = dmf_res["ts_guess"]
else:
    print("An initial TS guess was given, so DMF is skipped.")
    ts_guess = guess0

In [ ]:
#@title 7. TS optimization (Sella) and checks { display-mode: "form" }
ts_res = ts.run_sella(ts_guess, calc, job, exact_hessian=ts_exact_hessian)
ts_vib = ts.check_ts(ts_res, calc)
conn = None
if check_connection and ts_vib is not None and ts_vib["n_imag"] >= 1:
    conn = ts.connect(ts_res["atoms"], ts_vib, calc, job, reactant, product)
files, ts_log, uma_rows = ts.report(job, reactant, product, dmf_res, ts_res, ts_vib, conn, endpoints)
files["ts.gjf"] = gaussian.gjf(ts_res["atoms"], job, gaussian_route, f"{job.out_prefix} TS from ColabUMA ({model_name})",
                                 chk=f"{job.out_prefix}_ts.chk")

In [ ]:
#@title 8. DFT single points { display-mode: "form" }
dft_log = []
if run_dft:
    _f, dft_log = dft.run_optional(ts.dft_points(ts_res, endpoints, conn), charge, multiplicity, dft_functional,
                                   dft_basis, uma_rows, dft_time_limit)
    files.update(_f)
else:
    print("DFT single points are off.")

In [ ]:
#@title 9. Save { display-mode: "form" }
_cond = [f"Initial TS: {ts_input}", f"Endpoint optimization: {'on' if (use_dmf and opt_endpoints) else 'off'}",
         f"Sella initial Hessian: {'numerical' if ts_exact_hessian else 'approximate'}", f"Simple IRC check: {'on' if check_connection else 'off'}"]
files["log.txt"] = output.log_text("ColabUMA TS log", source_desc, job, [_cond, ts_log, dft_log])
zip_path = output.save_zip(job.out_prefix, files)